<a href="https://colab.research.google.com/github/Nivitha29/Neural-Network---Practice/blob/main/multiple_inputsOutputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this module -**

Learining to create neural network with multiple input and output with lightningModule. this is a simple classiification model to prediction the flower type from Petal and sepal width.

In [ ]:
%%capture

!pip install lightning

### Understanding `import torch` and Tensors

- **`import torch`**: This line imports the PyTorch library. PyTorch is an open-source machine learning framework that is widely used for deep learning tasks. It provides tools for building and training neural networks.

- **Tensors**: In PyTorch, a tensor is the fundamental data structure, similar to NumPy arrays but with the added capability to run on GPUs (Graphics Processing Units) for accelerated computation. Tensors are multi-dimensional arrays that can store numerical data (e.g., scalars, vectors, matrices, or higher-dimensional arrays). They are crucial for deep learning as they are used to represent all inputs, outputs, and parameters of a neural network.

In [ ]:
import torch # torch will allow us to create tensors.
import torch.nn as nn # torch.nn allows us to create a neural network.
import torch.nn.functional as F # nn.functional give us access to the activation and loss functions.
from torch.optim import Adam # optim contains many optimizers. This time we're using Adam

import lightning as L # lightning has tons of cool tools that make neural networks easier
from torch.utils.data import TensorDataset, DataLoader # these are needed for the training data

import pandas as pd # We'll use pandas to read in the data and normalize it
from sklearn.model_selection import train_test_split # We'll use this to create training and testing datasets


Importing dataset - using Iris dataset

In [ ]:
## We'll read in the dataset with the pandas function read_table()
## read_table() can read in various text files including, comma-separated and tab-delimted.

# The original line caused a FileNotFoundError because 'iris.txt' was not found.
# url = "./iris.txt"
# df = pd.read_table(url, sep=",", header=None)

# Instead, we'll load the Iris dataset directly from sklearn and convert it to a DataFrame.
from sklearn.datasets import load_iris

## NOTE: If the data were tab-delimted, we would set sep="\t".
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target

# looking into the dataset
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [ ]:
## To name each column, we assign a list of column names to `columns`
df.columns = ["sepal_length",
              "sepal_width",
              "petal_length",
              "petal_width",
              "class"]

## To verify we did that correctly, let's print out the first few rows
df.head()

df.shape ## shape returns the rows and colunns...

## To determine the number of iris species in the dataset,
## we'll count the number of unique values in the column called `class`.
df['class'].nunique()

## We can print out the unique values in a dataframe's column with the 'unique()' method.
df['class'].unique()

array([np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')],
      dtype=object)

In [ ]:
# The 'target' attribute of the iris dataset contains the names for classes 0, 1, and 2.
# Let's map these to our 'class' column.
class_names = iris.target_names

# Create a dictionary to map numerical classes to their names
class_mapping = {i: name for i, name in enumerate(class_names)}

# Apply the mapping to the 'class' column
df['class'] = df['class'].replace(class_mapping)

print("DataFrame after renaming class labels:")
display(df.head())


DataFrame after renaming class labels:


,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


To find out: How many rows per type

In [ ]:
for class_name in df['class'].unique(): # for each unique class name...

    ## ...print out the number of rows associated with it
    print(class_name, ": ", sum(df['class'] == class_name), sep="")

setosa: 50
versicolor: 50
virginica: 50


Now, let's split the data into training and testing datasets. The first step is to separate the columns into input values and labels.

In this example, to keep the neural network simple, we'll just use petal_width and sepal_width values for the inputs. So the first we'll do is make sure we can correctly isolate the columns we want from the columns we don't want. We do this by passing df a list of column names we want to get values for, ['petal_width', 'sepal_width'].

In [ ]:
input_values = df[['petal_width', 'sepal_width']]
input_values.head()

label_values = df['class']
label_values.head()

## Convert the strings in the 'class' column into numbers with factorize()
classes_as_numbers = label_values.factorize()[0] ## NOTE: factorize() returns a list of lists,
                                                 ## and since we only need the first list of values,
                                                 ## we index the output of factorize() with [0].
classes_as_numbers


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

Now we need to split the dataset for training and testing. use the function train_test_split() from sklearn

In [ ]:
input_train, input_test, label_train, label_test = train_test_split(input_values,
                                                                    classes_as_numbers,
                                                                    test_size=0.25,
                                                                    stratify=classes_as_numbers,
                                                                    random_state=42)

# verify train and test shape
input_train.shape


(112, 2)

In [ ]:
label_train.shape

(112,)

In [ ]:
label_test.shape

(38,)

In [ ]:
input_test.shape

(38, 2)

Create one hot encoding for each flower type
eg - setosa [1,0,0] ,
Virginica [0,1,0]

In [ ]:
## Now create a new tensor with one-hot encoded rows for each row in the original dataset.
one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

## Print out a few of the rows one-hot encoded data.
one_hot_label_train[:10]

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]])

The next step is normalizing the data (0-1)

In [ ]:
## First, determine the maximum values in input_train...
max_vals_in_input_train = input_train.max()
## Now print them out...
max_vals_in_input_train

,0
petal_width,2.5
sepal_width,4.4


In [ ]:
## Second, determine the minimum values in input_train
min_vals_in_input_train = input_train.min()
## Now print them out...
min_vals_in_input_train

,0
petal_width,0.1
sepal_width,2.0


In [ ]:
## Now normalize input_train with the maximum and minimum values from input_train
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_train.head()

,petal_width,sepal_width
130,0.750000,0.333333
122,0.791667,0.333333
81,0.375000,0.166667
71,0.500000,0.333333
89,0.500000,0.208333


In [ ]:
## Now normalize input_test with the maximum and minimum values from input_train
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test.head()

,petal_width,sepal_width
42,0.041667,0.500000
56,0.625000,0.541667
99,0.500000,0.333333
53,0.500000,0.125000
38,0.041667,0.416667


Converting dataframe into Tensor for neural network

In [ ]:
## Convert the DataFrame input_train into tensors
input_train_tensors = torch.tensor(input_train.values).type(torch.float32)

## now print out the first 5 rows to make sure they are what we expect.
input_train_tensors[:5]

tensor([[0.7500, 0.3333],
        [0.7917, 0.3333],
        [0.3750, 0.1667],
        [0.5000, 0.3333],
        [0.5000, 0.2083]])

In [ ]:
## Convert the DataFrame input_test into tensors
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)

## now print out the first 5 rows to make sure they are what we expect.
input_test_tensors[:5]

tensor([[0.0417, 0.5000],
        [0.6250, 0.5417],
        [0.5000, 0.3333],
        [0.5000, 0.1250],
        [0.0417, 0.4167]])

In [ ]:
train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

Building a neural network with PyTorch means creating a new class. And to make it easy to train the neural network, this class will inherit from LightningModule.

Our new class will have the following methods:

__init__() to initialize the Weights and Biases and keep track of a few other housekeeping things.
forward() to make a forward pass through the neural network.
configure_optimizers() to configure the optimizer. There are lots of optimizers to choose from, but in this tutorial, we'll change things up and use Adam.
training_step() to pass the training data to forward(), calculate the loss and keep track of the loss values in a log file.

In [ ]:
class MultipleInsOuts(L.LightningModule):

    def __init__(self):

        super().__init__() ## We call the __init__() for the parent, LightningModule, so that it
                           ## can initialize itself as well.

        L.seed_everything(seed=42)

        ############################################################################
        ##
        ## Here is where we initialize the Weights and Biases for the neural network
        ##
        ############################################################################

        ## If you look at the drawing of the network we want to build (above),
        ## you see that we have 2 inputs that lead to 2 activation functions.
        ## We create these connections and initialize their Weights and Biases
        ## with the nn.Linear() function by setting in_features=2 and out_features=2.
        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)

        ## Next, we see that the 2 activation functions are connected to 3 outputs.
        ## We create these connections and initialize their Weights and Biases
        ## with the nn.Linear() function by setting in_features=2 and out_features=3.
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)

        self.loss = nn.MSELoss(reduction='sum')


    def forward(self, input):
        ## First, we run the input values to the activation functions
        ## in the hidden layer
        hidden = self.input_to_hidden(input)
        ## Then we run the values through a ReLU activation function
        ## and then run those values to the output.
        output_values = self.hidden_to_output(torch.relu(hidden))

        return(output_values)


    def configure_optimizers(self):
        ## In this example, configuring the optimizer
        ## consists of passing it the weights and biases we want
        ## to optimize, which are all in self.parameters(),
        ## and setting the learning rate with lr=0.001.
        return Adam(self.parameters(), lr=0.001)


    def training_step(self, batch, batch_idx):
        ## The first thing we do is split 'batch'
        ## into the input and label values.
        inputs, labels = batch

        ## Then we run the input through the neural network
        outputs = self.forward(inputs)

        ## Then we calculate the loss.
        loss = self.loss(outputs, labels)

        ## Lastly, we could add the loss a log file
        ## so that we can graph it later. This would
        ## help us decide if we have done enough training
        ## Ideally, if we do enough training, the loss
        ## should be small and not getting any smaller.
        # self.log("loss", loss)

        return loss

Now the class is created MultipleInsOuts, training the model

In [ ]:
model = MultipleInsOuts()

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


In [ ]:
trainer = L.Trainer(max_epochs=10)
trainer.fit(model, train_dataloaders=train_dataloader)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ input_to_hidden  │ Linear  │      6 │ train │     0 │
│ 1 │ hidden_to_output │ Linear  │      9 │ train │     0 │
│ 2 │ loss             │ MSELoss │      0 │ train │     0 │
└───┴──────────────────┴─────────┴────────┴───────┴───────┘

Trainable params: 15                                                                                               
Non-trainable params: 0                                                                                            
Total params: 15                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [ ]:
# Run the input_test_tensors through the neural network
predictions = model(input_test_tensors)

In [ ]:
predictions[0:4,]

tensor([[0.7582, 0.0283, 0.0325],
        [0.1993, 0.3881, 0.4945],
        [0.2478, 0.3494, 0.3868],
        [0.1938, 0.3775, 0.3712]], grad_fn=<SliceBackward0>)

We can determine which species was predicted in predictions by selecting the index in each row that corresponding to the largest value, and we do that with torch.argmax(). torch.argmax() returns a tensor that contains the indices with the largest values for each row.

In [ ]:
## Select the output with highest value...
predicted_labels = torch.argmax(predictions, dim=1) ## dim=0 applies argmax to rows, dim=1 applies argmax to columns
predicted_labels[0:4] # print out the first 4 predictions

tensor([0, 2, 2, 1])

In the first row index 0 had the largest value. Thus, the first prediction corresponds to Setosa. The second, third, and fourth rows predicted 2, which corresponds to Virginica.

Now, let's compare what the neural network predicted in predicted_labels to the known values in label_test and calculate the percentage of correct predictions. We do this by adding up the number of times an element in predicted_labels equals the corresponding element in label_test and dividing by the number of elements in predicted_labels.

In [ ]:
## Now compare predicted_labels with test_labels to calculate accuracy
## NOTE: torch.eq() computes element-wise equality between two tensors.
##       label_test, however, is just an array, so we convert it to a tensor
##       before passing it in. torch.sum() then adds up all of the "True"
##       output values to get the number of correct predictions.
##       We then divide the number of correct predictions by the number of predicted values,
##       obtained with len(predicted_labels), to get the percentage of correct predictions
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.7368)

correctly predicts only **74%** . Can you increase this by increasing the number of epochs? - this is the advantage of Lightning module, we do not need to start from scratch, Lightning stores the checkpoint files ( weight and parameter for each epoch), we can directly start from where it ended.





In [ ]:
# First identify the checkpoint file
path_to_checkpoint = trainer.checkpoint_callback.best_model_path ## By default, "best" = "most recent"

## First, create a new Lightning Trainer
trainer = L.Trainer(max_epochs=100) # Before, max_epochs=10, so, by setting it to 100, we're adding 90 more.

## Then call trainer.fit() using the path to the most recent checkpoint files
## so that we can pick up where we left off.
trainer.fit(model, train_dataloaders=train_dataloader, ckpt_path=path_to_checkpoint)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ input_to_hidden  │ Linear  │      6 │ train │     0 │
│ 1 │ hidden_to_output │ Linear  │      9 │ train │     0 │
│ 2 │ loss             │ MSELoss │      0 │ train │     0 │
└───┴──────────────────┴─────────┴────────┴───────┴───────┘

Trainable params: 15                                                                                               
Non-trainable params: 0                                                                                            
Total params: 15                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

INFO: Restored all states from the checkpoint at /content/lightning_logs/version_0/checkpoints/epoch=9-step=1120.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restored all states from the checkpoint at /content/lightning_logs/version_0/checkpoints/epoch=9-step=1120.ckpt


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


In [ ]:
# Run the input_test_tensors through the neural network
predictions = model(input_test_tensors)

## Select the output with highest value...
predicted_labels = torch.argmax(predictions, dim=1) ## dim=0 applies softmax to rows, dim=1 applies softmax to columns

## Now compare predicted_labels with test_labels to calculate accuracy
## NOTE: torch.eq() computes element-wise equality between two tensors.
##       label_test, however, is just an array, so we convert it to a tensor
##       before passing it in. torch.sum() then adds up all of the "True"
##       output values to get the number of correct predictions.
##       We then divide the number of correct predictions by the number of predicted values,
##       obtained with len(predicted_labels), to get the percentage of correct predictions
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.9211)

Now the model correctly classified **92%** of the testing dataset

We can also make prediction by submitting new input values

In [ ]:
normalized_values = ([0.2, 3.0] - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
normalized_values



/tmp/ipykernel_1913/4290610102.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model(torch.tensor(normalized_values).type(torch.float32))


tensor([ 0.7876,  0.2068, -0.0427], grad_fn=<ViewBackward0>)

In [ ]:
model(torch.tensor(normalized_values).type(torch.float32))

/tmp/ipykernel_1913/3219751161.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model(torch.tensor(normalized_values).type(torch.float32))


tensor([ 0.7876,  0.2068, -0.0427], grad_fn=<ViewBackward0>)

So the first element is closer to 1 tensor([ 0.7876,  0.2068, -0.0427]
, therefore it is Setosa!!